# Ask the Bug Report: Chatbot (Jupyter)

**Course:** AI for Cybersecurity and Digital Forensics  ·  **SectraTEK Lab**

The same bug-report agent, run **in the notebook**: call `ask("...")` and read the agent's
reasoning and cited section. A **bug report** is the output of `adb bugreport` - a big text file
split into sections (device info, packages, battery, Wi-Fi, accounts, logcat). The agent has
three tools: list the sections, read a section, or search the whole report.

### 1. Install the client and write the code + sample report

In [ ]:
%pip install -q openai

In [ ]:
%%writefile agentkit.py
"""
agentkit.py - the shared forensic-agent core used by the Lab 4 notebooks and chatbots.

One place defines: how to connect to a model (OpenRouter or Ollama), how to build the
synthetic Android image, the read-only tools the agent may call, the think-act-observe
loop, and a conversational wrapper for the chatbots. The chatbot .py apps import this
module; the teaching notebooks embed the same code inline so students open only one file.
"""
import json
import os
import re
import sqlite3
from pathlib import Path


# ======================================================================= model backend
def connect_model(provider="openrouter", model=None, api_key=None, base_url=None):
    """Return (client, model_id). provider is 'openrouter' or 'ollama'.

    openrouter: free hosted Qwen, needs a key from https://openrouter.ai/keys.
    ollama:     a Qwen model running locally (https://ollama.com), no key, nothing leaves the PC.
    """
    from openai import OpenAI
    provider = provider.lower()
    if provider == "ollama":
        base_url = base_url or "http://localhost:11434/v1"
        client = OpenAI(base_url=base_url, api_key="ollama")  # Ollama ignores the key
        return client, (model or "qwen3:8b")
    # default: OpenRouter
    base_url = base_url or "https://openrouter.ai/api/v1"
    api_key = api_key or os.environ.get("OPENROUTER_API_KEY", "")
    client = OpenAI(base_url=base_url, api_key=api_key)
    return client, (model or "qwen/qwen3-30b-a3b:free")


# ======================================================================= synthetic image
def build_image(root="android_image"):
    """Create a tiny, synthetic Android extraction on disk. Everything in it is fictional."""
    ROOT = Path(root)
    text_files = {
        f"{root}/build.prop":
            "# Synthetic build.prop (teaching data, fictional)\n"
            "ro.product.manufacturer=Google\nro.product.brand=google\n"
            "ro.product.model=Pixel 7a\nro.product.device=lynx\nro.product.name=lynx\n"
            "ro.build.version.release=14\nro.build.version.sdk=34\n"
            "ro.build.fingerprint=google/lynx/lynx:14/AP2A.240705.005/2024072800:user/release-keys\n",
        f"{root}/data/system/packages.list":
            "com.google.android.gm 10089 0 /data/user/0/com.google.android.gm default 0 1\n"
            "com.android.providers.contacts 10003 0 /data/user/0/com.android.providers.contacts platform 0 1\n"
            "com.android.providers.telephony 10004 0 /data/user/0/com.android.providers.telephony platform 0 1\n"
            "com.whatsapp 10137 0 /data/user/0/com.whatsapp default 0 1\n"
            "com.instagram.android 10140 0 /data/user/0/com.instagram.android default 0 1\n"
            "org.thoughtcrime.securesms 10142 0 /data/user/0/org.thoughtcrime.securesms default 0 1\n",
        f"{root}/data/system/users/0/settings_secure.xml":
            "<?xml version='1.0' encoding='utf-8' standalone='yes' ?>\n<settings version=\"241\">\n"
            "  <setting id=\"1\" name=\"android_id\" value=\"a1b2c3d4e5f60718\" package=\"android\" />\n"
            "  <setting id=\"2\" name=\"bluetooth_name\" value=\"Pixel 7a\" package=\"com.android.settings\" />\n"
            "  <setting id=\"3\" name=\"device_name\" value=\"Logan's Pixel 7a\" package=\"com.android.settings\" />\n"
            "</settings>\n",
        f"{root}/data/misc/wifi/WifiConfigStore.xml":
            "<?xml version='1.0' encoding='utf-8' standalone='yes' ?>\n<WifiConfigStoreData>\n  <NetworkList>\n"
            "    <Network><string name=\"SSID\">&quot;HomeNet_5G&quot;</string>"
            "<long name=\"LastConnectedTime\" value=\"1721295600000\" /></Network>\n"
            "    <Network><string name=\"SSID\">&quot;BrightBeans Cafe&quot;</string>"
            "<long name=\"LastConnectedTime\" value=\"1720950000000\" /></Network>\n"
            "    <Network><string name=\"SSID\">&quot;Logan-Office&quot;</string>"
            "<long name=\"LastConnectedTime\" value=\"1721210400000\" /></Network>\n"
            "  </NetworkList>\n</WifiConfigStoreData>\n",
        # a JSON artifact (many Android apps store prefs/state as JSON)
        f"{root}/data/data/com.whatsapp/files/me.json":
            json.dumps({"jid": "15550143788@s.whatsapp.net", "pushname": "Logan D",
                        "registered": True, "last_backup": "2024-07-18"}, indent=2),
    }
    for rel, content in text_files.items():
        p = Path(rel)
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(content)

    def make_db(rel, schema, inserts):
        p = Path(rel)
        p.parent.mkdir(parents=True, exist_ok=True)
        if p.exists():
            p.unlink()
        con = sqlite3.connect(p)
        con.executescript(schema)
        for sql, params in inserts:
            con.executemany(sql, params)
        con.commit()
        con.close()

    make_db(f"{root}/data/data/com.google.android.gm/databases/accounts.db",
            "CREATE TABLE accounts(name TEXT, type TEXT, last_sync INTEGER);",
            [("INSERT INTO accounts VALUES (?,?,?)",
              [("ldehner505@gmail.com", "com.google", 1721295600),
               ("+15550143788", "com.whatsapp", 1721240000)])])
    make_db(f"{root}/data/data/com.android.providers.contacts/databases/contacts2.db",
            "CREATE TABLE contacts(display_name TEXT, phone TEXT, times_contacted INTEGER);",
            [("INSERT INTO contacts VALUES (?,?,?)",
              [("Marcus Webb", "+15550142210", 31), ("Priya Shah", "+15550148890", 12),
               ("Dad", "+15550149001", 54), ("BrightBeans Cafe", "+15550140077", 3),
               ("Alina K (work)", "+15550147325", 20)])])
    make_db(f"{root}/data/data/com.android.providers.telephony/databases/mmssms.db",
            "CREATE TABLE sms(address TEXT, date INTEGER, body TEXT, type INTEGER);",
            [("INSERT INTO sms VALUES (?,?,?,?)",
              [("+15550142210", 1721200000000, "Are we still on for Friday?", 1),
               ("+15550142210", 1721200600000, "Yes, 6pm at the usual place.", 2),
               ("+15550149001", 1721240500000, "Landed safe, call you tonight.", 2),
               ("+15550148890", 1721250000000, "Can you send the invoice?", 1),
               ("+15550149001", 1721299000000, "Thanks Dad, talk soon.", 2)])])
    make_db(f"{root}/data/data/com.whatsapp/databases/msgstore.db",
            "CREATE TABLE messages(chat TEXT, timestamp INTEGER, from_me INTEGER, text TEXT);",
            [("INSERT INTO messages VALUES (?,?,?,?)",
              [("Marcus Webb", 1721260000000, 0, "Did the package arrive?"),
               ("Marcus Webb", 1721260400000, 1, "Not yet, will check tomorrow."),
               ("Priya Shah", 1721280000000, 1, "Sent the files to your email.")])])
    return ROOT


# ======================================================================= the tools
# A text-ish file is one the agent can read as text. Everything else is "binary".
_TEXT_EXT = {".txt", ".xml", ".json", ".prop", ".list", ".log", ".csv", ".ini", ".conf", ".html"}


def make_tools(root="android_image", allow_python=False):
    """Build the read-only toolbox, all sealed inside `root`. Returns {name: function}."""
    ROOT = Path(root)

    def _resolve(path):
        base = ROOT.resolve()
        path = (path or ".").strip().lstrip("/")
        target = base if path in ("", ".") else (ROOT / path).resolve()
        if target != base and base not in target.parents:
            raise ValueError("path is outside the image")
        return target

    def list_dir(path="."):
        """List what is inside a folder (folders shown with a trailing '/')."""
        t = _resolve(path)
        if not t.exists():
            return f"No such path: {path}"
        if t.is_file():
            return f"{path} is a file, not a folder. Use read_file or run_sql."
        items = [e + ("/" if (t / e).is_dir() else "") for e in sorted(os.listdir(t))]
        return "\n".join(items) if items else "(empty folder)"

    def read_file(path, max_chars=1500):
        """Read a file. Pretty-prints JSON; for SQLite databases points you to run_sql;
        for other binary files shows the type and the first bytes."""
        t = _resolve(path)
        if not t.exists() or t.is_dir():
            return f"No such file: {path}"
        head = t.read_bytes()[:16]
        if head[:15] == b"SQLite format 3":
            con = sqlite3.connect(f"file:{t}?mode=ro", uri=True)
            tables = [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table'")]
            con.close()
            return f"[{path} is a SQLite database. Tables: {tables}. Use run_sql with a SELECT query.]"
        ext = t.suffix.lower()
        if ext not in _TEXT_EXT and b"\x00" in head:
            return (f"[{path} is a binary file ({t.stat().st_size} bytes). First bytes (hex): "
                    f"{head.hex(' ')}. It is not readable as text.]")
        text = t.read_text(errors="replace")
        if ext == ".json":
            try:
                text = json.dumps(json.loads(text), indent=2)
            except Exception:
                pass
        return text[:max_chars] + ("" if len(text) <= max_chars else "\n...[truncated]")

    def run_sql(path, query):
        """Run ONE read-only SQL SELECT against a SQLite database and return the rows."""
        t = _resolve(path)
        if not t.exists():
            return f"No such database: {path}"
        q = (query or "").strip().rstrip(";")
        if not q.lower().startswith(("select", "pragma")):
            return "Only read-only SELECT queries are allowed."
        try:
            con = sqlite3.connect(f"file:{t}?mode=ro", uri=True)
            cur = con.execute(q)
            cols = [c[0] for c in cur.description] if cur.description else []
            rows = cur.fetchmany(50)
            con.close()
        except Exception as e:
            return f"SQL error: {e}"
        if not rows:
            return "(no rows)"
        return "\n".join([" | ".join(cols)] + [" | ".join(str(c) for c in r) for r in rows])

    def find_files(name):
        """Find files anywhere in the image whose path contains `name` (case-insensitive)."""
        base = ROOT.resolve()
        needle = (name or "").lower()
        hits = []
        for p in sorted(base.rglob("*")):
            if p.is_file() and needle in str(p.relative_to(base)).lower():
                hits.append(str(p.relative_to(base)))
        if not hits:
            return f"No files matched {name!r}."
        return "\n".join(hits[:50]) + ("" if len(hits) <= 50 else f"\n...({len(hits)} total)")

    def search_text(keyword, max_hits=20):
        """Search the text inside readable files for `keyword` (case-insensitive).
        Returns path:line for each match, so you know which file to open next."""
        base = ROOT.resolve()
        needle = (keyword or "").lower()
        out = []
        for p in sorted(base.rglob("*")):
            if not p.is_file() or p.suffix.lower() not in _TEXT_EXT:
                continue
            try:
                for i, line in enumerate(p.read_text(errors="replace").splitlines(), 1):
                    if needle in line.lower():
                        out.append(f"{p.relative_to(base)}:{i}: {line.strip()[:120]}")
                        if len(out) >= max_hits:
                            return "\n".join(out) + "\n...[more matches not shown]"
            except Exception:
                continue
        return "\n".join(out) if out else f"No text matches for {keyword!r}."

    tools = {"list_dir": list_dir, "read_file": read_file, "run_sql": run_sql,
             "find_files": find_files, "search_text": search_text}

    if allow_python:
        def run_python(code):
            """ADVANCED, OPTIONAL: run a short read-only Python snippet. `ROOT` (the image folder,
            a pathlib.Path) is available. Print what you want to see. Writing to files is blocked."""
            import io
            import contextlib
            if any(bad in code for bad in ("open(", "write", "unlink", "remove", "rmtree",
                                           "os.system", "subprocess", "Popen", "__import__")):
                return "Blocked: that snippet looks like it writes or runs external commands."
            buf = io.StringIO()
            safe = {"ROOT": ROOT, "sqlite3": sqlite3, "json": json, "Path": Path, "re": re}
            try:
                with contextlib.redirect_stdout(buf):
                    exec(code, safe)  # teaching tool: sandboxed namespace, read-only guard above
            except Exception as e:
                return f"Python error: {e}\n{buf.getvalue()}"
            out = buf.getvalue().strip()
            return out[:1500] if out else "(the snippet ran but printed nothing)"
        tools["run_python"] = run_python

    return tools


def tool_descriptions(tools):
    """One line per tool, from its docstring, for the system prompt."""
    sigs = {
        "list_dir": "list_dir(path)", "read_file": "read_file(path)",
        "run_sql": "run_sql(path, query)", "find_files": "find_files(name)",
        "search_text": "search_text(keyword)", "run_python": "run_python(code)",
    }
    lines = []
    for name, fn in tools.items():
        doc = (fn.__doc__ or "").strip().split("\n")[0]
        lines.append(f"- {sigs.get(name, name)}: {doc}")
    return "\n".join(lines)


# ======================================================================= the agent
def build_system_prompt(tools):
    return (
        "You are Sectra, a digital-forensics examiner's assistant. You are investigating an Android "
        "device extraction (a copy of the phone's files) to answer the examiner's question. You cannot "
        "see any file until you open it with a tool, and you must base every answer only on what the "
        "tools return.\n\n"
        "TOOLS (use exactly one per step):\n" + tool_descriptions(tools) + "\n\n"
        "HOW TO REPLY: send ONE JSON object and nothing else.\n"
        "To use a tool:\n"
        '  {"thought": "<one short sentence>", "action": "list_dir", "action_input": {"path": "data/data"}}\n'
        "When you can answer, reply:\n"
        '  {"thought": "<one short sentence>", "final_answer": "<your answer>", "evidence_path": "<the file you used>"}\n\n'
        "RULES: investigate step by step; use ONLY what the tools return; never invent paths, tables, "
        "or data; if a tool errors, try another path or tool; if the evidence is not there, say so "
        "plainly; always cite the exact file in evidence_path. Keep thoughts to one sentence. /no_think"
    )


def parse_action(raw):
    """Pull the JSON object out of the model's reply, tolerating <think> blocks and code fences."""
    text = re.sub(r"<think>.*?</think>", "", raw, flags=re.S)
    text = text.replace("```json", "").replace("```", "")
    start = text.find("{")
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(text[start:i + 1])
                except Exception:
                    return None
    return None


def run_agent_iter(question, client, model, tools, system_prompt=None, max_steps=12, history=None):
    """Generator version of the agent, for live UIs. Yields a dict per step
    ({step, thought, action, args, observation}) and finally the answer
    ({final_answer, evidence_path}) or a give-up dict ({final_answer: None}).

    history: optional prior [(user, assistant)] turns, so the agent remembers a conversation.
    """
    system_prompt = system_prompt or build_system_prompt(tools)
    messages = [{"role": "system", "content": system_prompt}]
    for u, a in (history or []):
        messages.append({"role": "user", "content": u})
        messages.append({"role": "assistant", "content": a})
    messages.append({"role": "user", "content": f"Investigate and answer this question: {question}"})

    for step in range(1, max_steps + 1):
        raw = client.chat.completions.create(
            model=model, messages=messages, temperature=0).choices[0].message.content
        messages.append({"role": "assistant", "content": raw})
        move = parse_action(raw)
        if move is None:
            messages.append({"role": "user",
                             "content": "That was not valid JSON. Reply with ONE JSON object only."})
            continue
        if "final_answer" in move:
            yield {"final_answer": move["final_answer"], "evidence_path": move.get("evidence_path", "")}
            return
        tool, args = move.get("action"), move.get("action_input", {}) or {}
        if tool not in tools:
            observation = f"Unknown tool {tool!r}. Use one of: {', '.join(tools)}."
        else:
            try:
                observation = tools[tool](**args)
            except Exception as e:
                observation = f"Tool error: {e}"
        yield {"step": step, "thought": move.get("thought", ""),
               "action": tool, "args": args, "observation": observation}
        messages.append({"role": "user", "content": f"Observation:\n{observation}"})
    yield {"final_answer": None, "evidence_path": ""}


def run_agent(question, client, model, tools, system_prompt=None, max_steps=12,
              on_step=None, history=None):
    """Drive the model think->act->observe until it answers.

    on_step(dict): optional callback for live UIs (gets each step dict, then the final dict).
    history: optional prior [(user, assistant)] turns, so the agent remembers a conversation.
    Returns the final move dict (with 'final_answer'/'evidence_path'), or None if it gave up.
    """
    final = None
    for ev in run_agent_iter(question, client, model, tools, system_prompt, max_steps, history):
        if on_step:
            on_step(ev)
        if "final_answer" in ev:
            final = ev if ev["final_answer"] is not None else None
    return final


def answer_text(move):
    """Format a final move as a human answer line with its citation."""
    if not move:
        return "I could not reach an answer within the step limit. Try rephrasing the question."
    cite = move.get("evidence_path") or "(no file cited)"
    return f"{move['final_answer']}\n\nSource: {cite}"


In [ ]:
%%writefile bugkit.py
"""
bugkit.py - a forensic agent over an Android BUG REPORT (adb bugreport).

A bug report is one big sectioned text file (device info, packages, battery, wifi,
accounts, logcat, ...). This reuses the same agent loop as agentkit, but gives it tools
that navigate a bug report instead of a filesystem image: list its sections, read a
section, or search it. Students can point it at their OWN `adb bugreport` output.
"""
import re
import zipfile
from pathlib import Path

import agentkit as ak  # reuse connect_model, parse_action, run_agent, run_agent_iter, answer_text

connect_model = ak.connect_model
run_agent = ak.run_agent
run_agent_iter = ak.run_agent_iter
answer_text = ak.answer_text

_SECTION_RE = re.compile(r"^-{3,}\s*(.+?)\s*-{3,}\s*$")


def _read_any(path):
    """Read a bug report from a .txt, or the first bugreport*.txt inside a .zip."""
    p = Path(path)
    if p.suffix.lower() == ".zip":
        with zipfile.ZipFile(p) as z:
            names = [n for n in z.namelist() if n.lower().endswith(".txt") and "bugreport" in n.lower()]
            names = names or [n for n in z.namelist() if n.lower().endswith(".txt")]
            if not names:
                raise ValueError("no .txt bug report found inside the zip")
            return z.read(names[0]).decode("utf-8", "replace")
    return p.read_text(errors="replace")


def load_report(path):
    """Return a list of (section_title, section_body) in order."""
    text = _read_any(path)
    sections, title, cur = [], "HEADER", []
    for line in text.splitlines():
        m = _SECTION_RE.match(line)
        if m:
            sections.append((title, "\n".join(cur)))
            title, cur = m.group(1).strip(), []
        else:
            cur.append(line)
    sections.append((title, "\n".join(cur)))
    return sections


def make_bug_tools(sections):
    """Three read-only tools over a parsed bug report."""
    def list_sections():
        """List the sections of the bug report, by number and title."""
        return "\n".join(f"{i}. {t}" for i, (t, _) in enumerate(sections))

    def read_section(name, max_chars=1800):
        """Read one section, found by its number or by text in its title."""
        key = (name or "").strip().lower()
        for i, (t, body) in enumerate(sections):
            if key == str(i) or (key and key in t.lower()):
                body = body.strip() or "(section is empty)"
                return f"[{t}]\n" + (body[:max_chars] + ("" if len(body) <= max_chars else "\n...[truncated]"))
        return f"No section matching {name!r}. Use list_sections to see the titles."

    def search_report(keyword, max_hits=25):
        """Search the whole bug report for a keyword; returns [section] matching lines."""
        key = (keyword or "").lower()
        out = []
        for t, body in sections:
            for line in body.splitlines():
                if key in line.lower():
                    out.append(f"[{t}] {line.strip()[:140]}")
                    if len(out) >= max_hits:
                        return "\n".join(out) + "\n...[more matches not shown]"
        return "\n".join(out) if out else f"No matches for {keyword!r}."

    return {"list_sections": list_sections, "read_section": read_section, "search_report": search_report}


def build_bug_system_prompt(tools):
    return (
        "You are Sectra, a digital-forensics examiner's assistant. You are reading an Android BUG REPORT "
        "(the output of `adb bugreport`): one large text file split into sections such as device "
        "properties, installed packages, battery, Wi-Fi, accounts, and the system log. You answer the "
        "examiner's question using ONLY what the tools return.\n\n"
        "TOOLS (use exactly one per step):\n" + ak.tool_descriptions(tools) + "\n\n"
        "HOW TO REPLY: send ONE JSON object and nothing else.\n"
        "To use a tool:\n"
        '  {"thought": "<one short sentence>", "action": "search_report", "action_input": {"keyword": "battery"}}\n'
        "When you can answer, reply:\n"
        '  {"thought": "<one short sentence>", "final_answer": "<your answer>", "evidence_path": "<the section you used>"}\n\n'
        "RULES: investigate step by step; a good first move is often search_report for a keyword, or "
        "list_sections to see what is there; use ONLY what the tools return; never invent data; if the "
        "answer is not in the report, say so; cite the section you used in evidence_path. /no_think"
    )


def quick_agent(question, client, model, sections, **kw):
    """Convenience: build tools + prompt for a report and run the agent."""
    tools = make_bug_tools(sections)
    return run_agent(question, client, model, tools, build_bug_system_prompt(tools), **kw)


In [ ]:
%%writefile sample_bugreport.txt
========================================================
== dumpstate: 2024-07-28 10:15:03
== (SYNTHETIC bug report, fictional, for teaching only)
========================================================

Build: google/lynx/lynx:14/AP2A.240705.005/2024072800:user/release-keys
Build fingerprint: 'google/lynx/lynx:14/AP2A.240705.005/2024072800:user/release-keys'
Bootloader: lynx-1.2-10000000
Radio: g5300q-000000
Network: Verizon, LTE
Kernel: Linux version 5.15.123-android14
Uptime: up 2 days, 4 hours, 11 minutes
Bugreport format version: 2.0
Serial number: 1A2B3C4D5E

------ SYSTEM PROPERTIES ------
[ro.product.manufacturer]: [Google]
[ro.product.model]: [Pixel 7a]
[ro.product.device]: [lynx]
[ro.build.version.release]: [14]
[ro.build.version.sdk]: [34]
[ro.serialno]: [1A2B3C4D5E]
[persist.sys.timezone]: [America/New_York]
[gsm.sim.operator.alpha]: [Verizon]

------ MEMORY INFO (/proc/meminfo) ------
MemTotal:        7904320 kB
MemFree:          612340 kB
MemAvailable:    3120044 kB

------ ACCOUNT MANAGER (dumpsys account) ------
Accounts: 2
  Account {name=ldehner505@gmail.com, type=com.google}
  Account {name=+15550143788, type=com.whatsapp}
User UserInfo{0:Logan's Pixel 7a:c13} running

------ PACKAGE SERVICE (dumpsys package) ------
Packages:
  Package [com.google.android.gm] (gmail)
    versionName=2024.07.14
    firstInstallTime=2023-01-11 09:12:44
  Package [com.whatsapp] (whatsapp)
    versionName=2.24.14.78
    firstInstallTime=2023-02-02 18:40:10
  Package [com.instagram.android] (instagram)
    versionName=342.0.0.0
    firstInstallTime=2023-03-19 21:05:33
  Package [com.facebook.katana] (facebook)
    versionName=468.0.0.0
    firstInstallTime=2024-05-30 08:22:19
  Package [org.thoughtcrime.securesms] (signal)
    versionName=7.11.4
    firstInstallTime=2024-06-14 12:01:55

------ BATTERY STATS (dumpsys batterystats) ------
Battery level: 73
Battery scale: 100
Battery status: discharging
Battery health: good
Estimated battery capacity: 4385 mAh
Screen on time: 5h 12m over last 24h

------ WIFI (dumpsys wifi) ------
Wi-Fi is enabled
mWifiInfo SSID: "HomeNet_5G", BSSID: a4:12:cd:00:11:22, RSSI: -47
Saved networks:
  "HomeNet_5G" WPA2
  "BrightBeans Cafe" OPEN
  "Logan-Office" WPA2

------ LOCATION (dumpsys location) ------
Last Known Locations:
  gps: 40.7128,-74.0060 acc=12m time=2024-07-28 09:58:11

------ SYSTEM LOG (logcat -d) ------
07-28 10:12:33.123  1456  1498 I ActivityManager: Start proc com.whatsapp for service
07-28 10:12:40.004  1456  2011 W PackageManager: Package com.example.sideloaded has not been verified
07-28 10:13:01.456  1456  2001 E ActivityManager: ANR in com.instagram.android (reason: Input dispatching timed out)
07-28 10:13:45.900  1456  1500 I BatteryService: level=73 status=discharging
07-28 10:14:02.117  2203  2203 E AndroidRuntime: FATAL EXCEPTION in com.example.sideloaded: java.lang.NullPointerException
07-28 10:14:30.661  1456  1489 W WifiService: Connected to "HomeNet_5G"

------ USAGE STATS (dumpsys usagestats) ------
In-use time (last 24h):
  com.instagram.android: 3h 41m
  com.whatsapp: 1h 12m
  com.google.android.gm: 0h 34m

------ DUMPSYS BATTERYSTATS CHECKIN ------
(omitted in this synthetic sample)


### 2. Load a bug report

By default we use the bundled synthetic sample. **To use your own:** upload your `adb bugreport`
file (in Colab, use the file pane on the left; locally, put it next to this notebook) and set
`REPORT_PATH` to its name.

In [ ]:
import bugkit as bk
REPORT_PATH = 'sample_bugreport.txt'   # <- change to your own .txt or .zip bug report
sections = bk.load_report(REPORT_PATH)
print(f'Loaded {len(sections)} sections from {REPORT_PATH}:')
print(bk.make_bug_tools(sections)['list_sections']())

### 3. Connect the model and wire up the agent

In [ ]:
import os, getpass
PROVIDER = 'openrouter'   # or 'ollama' for a local Qwen
if PROVIDER == 'openrouter' and not os.environ.get('OPENROUTER_API_KEY'):
    os.environ['OPENROUTER_API_KEY'] = getpass.getpass('Paste your OpenRouter API key (hidden): ').strip()
client, MODEL = bk.connect_model(PROVIDER)
print('Ready. Model:', MODEL)

### 4. Ask questions

In [ ]:
def ask(question):
    print('You:', question, '\n')
    def show(ev):
        if 'final_answer' in ev:
            print(f"\nSectra: {ev['final_answer']}")
            print(f"Section: {ev.get('evidence_path') or '(none cited)'}")
        else:
            print(f"  [{ev['step']}] {ev['action']}({ev['args']}) -> {ev['observation'][:80].strip()}")
    return bk.quick_agent(question, client, MODEL, sections, on_step=show)

_ = ask('What phone model and Android version is this?')

### 5. Your turn

Try: *Who is the account owner?*  ·  *Which apps are installed?*  ·  *What is the battery level?*
·  *Are there any crashes in the system log?*

In [ ]:
_ = ask('Are there any errors or crashes in the system log?')